# 2. Feature Engineering

The goal of this section is to prepare the Bosch manufacturing dataset for machine learning.

The feature engineering process focuses on:

- Loading the numeric manufacturing dataset
- Removing low-information features
- Handling missing values
- Selecting useful features
- Creating a modeling-ready dataset
- Saving processed data for model training

In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

## 2.1 Load Dataset

We begin by loading the numeric dataset used during EDA.

This file contains product IDs, manufacturing measurements, and the target variable `Response`.

In [21]:
train = pd.read_csv(
    "/Users/junghanlee/Downloads/1. Python Code/3. Manufacturing Defect Prediction/1. Data/raw/train_numeric.csv"
)

train.head()

,Id,L0_S0_F0,L0_S0_F2,L0_S0_F4,L0_S0_F6,L0_S0_F8,L0_S0_F10,L0_S0_F12,L0_S0_F14,L0_S0_F16,...,L3_S50_F4245,L3_S50_F4247,L3_S50_F4249,L3_S50_F4251,L3_S50_F4253,L3_S51_F4256,L3_S51_F4258,L3_S51_F4260,L3_S51_F4262,Response
0,4,0.030,-0.034,-0.197,-0.179,0.118,0.116,-0.015,-0.032,0.020,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,7,0.088,0.086,0.003,-0.052,0.161,0.025,-0.015,-0.072,-0.225,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,9,-0.036,-0.064,0.294,0.330,0.074,0.161,0.022,0.128,-0.026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,11,-0.055,-0.086,0.294,0.330,0.118,0.025,0.030,0.168,-0.169,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


## 2.2 Separate Features and Target

The target variable is `Response`.

The `Id` column is a unique product identifier and should not be used as a predictive feature.

In [22]:
X = train.drop(columns=["Id", "Response"])
y = train["Response"]

X.shape, y.shape

((1183747, 968), (1183747,))

## 2.3 Remove Sparse Features

Many features contain a high percentage of missing values because not every product passes through every production station.

Features with extremely high missing ratios may provide limited modeling value and can increase computational cost.

In this project, features with more than 95% missing values are removed.

In [23]:
missing_ratio = X.isnull().mean()

sparse_features = missing_ratio[
    missing_ratio > 0.95
].index

len(sparse_features)

492

In [24]:
X_reduced = X.drop(columns=sparse_features)

X_reduced.shape

(1183747, 476)

## 2.4 Remove Constant Features

Features with only one unique value do not help the model distinguish between defective and non-defective products.

These low-information features are removed.

In [25]:
nunique = X_reduced.nunique(dropna=False)

constant_features = nunique[
    nunique <= 1
].index

len(constant_features)

0

In [26]:
X_reduced = X_reduced.drop(columns=constant_features)

X_reduced.shape

(1183747, 476)

## 2.5 Missing Value Imputation

Missing values are handled using median imputation.

Median imputation is useful for manufacturing data because it is more robust to outliers than mean imputation.

In [27]:
median_values = X_reduced.median()

X_imputed = X_reduced.fillna(median_values)

X_imputed.isnull().sum().sum()

np.int64(0)

## 2.6 Feature Selection Using Correlation

The Bosch dataset contains hundreds of features.

To reduce dimensionality and improve training efficiency, we select features that show the strongest relationship with the target variable.

Correlation is calculated using a random sample to reduce computation time.

In [28]:
sample_size = 100000

sample_index = X_imputed.sample(
    n=sample_size,
    random_state=42
).index

X_sample = X_imputed.loc[sample_index]
y_sample = y.loc[sample_index]

X_sample.shape, y_sample.shape

((100000, 476), (100000,))

In [29]:
correlation = X_sample.corrwith(
    y_sample
).abs()

correlation.sort_values(
    ascending=False
).head(20)

/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


L1_S24_F1723    0.063798
L1_S24_F1695    0.035762
L1_S24_F1632    0.028471
L1_S24_F1758    0.028247
L1_S24_F1672    0.025117
L1_S24_F1846    0.024946
L3_S29_F3458    0.021109
L3_S29_F3351    0.021109
L1_S24_F1604    0.020777
L3_S29_F3470    0.018496
L3_S29_F3464    0.018496
L3_S30_F3704    0.018162
L1_S24_F1844    0.017562
L1_S24_F1581    0.014688
L0_S8_F144      0.014497
L1_S24_F1838    0.014470
L1_S24_F1565    0.014467
L1_S24_F1667    0.013720
L3_S30_F3794    0.013520
L3_S29_F3339    0.012702
dtype: float64

## 2.7 Select Top Features

The top 100 features with the highest absolute correlation with the target variable are selected for modeling.

This creates a smaller and more efficient dataset while preserving the most relevant signals found during EDA.

In [30]:
top_features = correlation.sort_values(
    ascending=False
).head(100).index

len(top_features)

100

In [31]:
X_selected = X_imputed[top_features]

X_selected.shape

(1183747, 100)

## 2.8 Train/Test Split

The dataset is split into training and testing sets.

Because the target variable is highly imbalanced, stratified sampling is used to preserve the same defect ratio in both sets.

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((946997, 100), (236750, 100), (946997,), (236750,))

In [33]:
y_train.value_counts(normalize=True) * 100

Response
0    99.4189
1     0.5811
Name: proportion, dtype: float64

In [34]:
y_train.value_counts(normalize=True) * 100

Response
0    99.4189
1     0.5811
Name: proportion, dtype: float64

In [35]:
y_test.value_counts(normalize=True) * 100

Response
0    99.418796
1     0.581204
Name: proportion, dtype: float64

## 2.9 Save Processed Data

The processed training and testing datasets are saved for the modeling notebook.

This allows the modeling step to start directly from clean, selected features.

In [36]:
X_train.to_csv("/Users/junghanlee/Downloads/1. Python Code/3. Manufacturing Defect Prediction/1. Data/X_train.csv", index=False)
X_test.to_csv("/Users/junghanlee/Downloads/1. Python Code/3. Manufacturing Defect Prediction/1. Data/X_test.csv", index=False)
y_train.to_csv("/Users/junghanlee/Downloads/1. Python Code/3. Manufacturing Defect Prediction/1. Data/y_train.csv", index=False)
y_test.to_csv("/Users/junghanlee/Downloads/1. Python Code/3. Manufacturing Defect Prediction/1. Data/y_test.csv", index=False)

In [37]:
print("Processed datasets saved successfully.")

Processed datasets saved successfully.


## 2.10 Feature Engineering Summary

### Completed Steps

- Removed product ID from modeling features
- Separated features and target variable
- Removed features with more than 95% missing values
- Removed constant features
- Filled missing values using median imputation
- Selected top 100 features based on correlation with the target
- Created stratified train/test split
- Saved processed datasets for modeling

### Next Steps

In the next notebook, we will:

1. Build a baseline classification model
2. Train Random Forest and XGBoost models
3. Evaluate model performance using classification metrics
4. Compare models based on recall, F1-score, ROC-AUC, and PR-AUC